# GWAS: PLINK (linear) vs GEMMA (LMM)

Runs both methods on 1000 Genomes chr22, then compares with QQ plots, Manhattan plots, and scatter plots. Need the chr22 VCF in `data/` or project root, plus PLINK and GEMMA installed (see README).

## 1. Setup and paths

In [ ]:
import os
import subprocess
import shutil
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUT_DIR = os.path.join(PROJECT_ROOT, "outputs")
VCF_FILENAME = "ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
VCF_CHR22 = os.path.join(DATA_DIR, VCF_FILENAME)
if not os.path.isfile(VCF_CHR22):
    VCF_CHR22 = os.path.join(PROJECT_ROOT, VCF_FILENAME)

USE_QUICK_RUN = True
SUBSET_REGION = "22:20000000-25000000"  
SUBSET_VCF = os.path.join(DATA_DIR, "chr22_5mb.vcf.gz")
PLINK_PREFIX = os.path.join(OUT_DIR, "chr22_5mb" if USE_QUICK_RUN else "chr22_1kg")
VCF_TO_USE = VCF_CHR22 

PLINK_PATH = None
GEMMA_PATH = None

def find_cmd(name, common_paths):
    out = shutil.which(name)
    if out:
        return out
    for p in common_paths:
        if os.path.isfile(p) and os.access(p, os.X_OK):
            return p
    return name

PLINK_CMD = PLINK_PATH if (PLINK_PATH and os.path.isfile(PLINK_PATH)) else find_cmd("plink", [
    "/opt/homebrew/bin/plink", "/usr/local/bin/plink",
    os.path.expanduser("~/bin/plink"),
])
GEMMA_CMD = GEMMA_PATH if (GEMMA_PATH and os.path.isfile(GEMMA_PATH)) else find_cmd("gemma", [
    "/opt/homebrew/bin/gemma", "/usr/local/bin/gemma",
    os.path.expanduser("~/bin/gemma"),
])
print("PLINK:", PLINK_CMD)
print("GEMMA:", GEMMA_CMD)
import sys
if sys.platform == "darwin" and "linux" in (PLINK_CMD or "").lower():
    raise SystemError("Use the macOS build of PLINK, not the linux one. Set PLINK_PATH to the plink2 binary.")
if PLINK_CMD == "plink":
    print("Set PLINK_PATH to your plink2 path if you installed it manually.")
if GEMMA_CMD == "gemma":
    print("GEMMA not found; install it or set GEMMA_PATH.")

for d in (DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)
print("Quick run (5 Mb):", USE_QUICK_RUN)
print("VCF path:", VCF_CHR22)
print("VCF exists:", os.path.isfile(VCF_CHR22))

PLINK: /Users/colecarter/Downloads/plink2_mac_20260228/plink2
GEMMA: gemma
  → Install GEMMA or set GEMMA_CMD to full path below
Data dir: /Users/colecarter/CSE284Project/data
Output dir: /Users/colecarter/CSE284Project/outputs
Quick run (5 Mb): True
VCF path: /Users/colecarter/CSE284Project/ALL.chr22.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
VCF exists: True


## 1b. Quick run: create 5 Mb subset

If USE_QUICK_RUN is True, this builds a small VCF so the rest runs in a few minutes. Run once; then downstream steps use it. Need bcftools or pysam.

In [ ]:
if not USE_QUICK_RUN:
    VCF_TO_USE = VCF_CHR22
    print("Using full chr22 VCF (this will be slow).")
else:
    if os.path.isfile(SUBSET_VCF):
        VCF_TO_USE = SUBSET_VCF
        print("Subset already exists:", SUBSET_VCF)
    else:
        bcftools = shutil.which("bcftools")
        if bcftools:
            t0 = time.time()
            r = subprocess.run([
                bcftools, "view", "-r", SUBSET_REGION, "-Oz", "-o", SUBSET_VCF,
                VCF_CHR22,
            ], capture_output=True, text=True, timeout=600)
            if r.returncode != 0:
                print("bcftools stderr:", r.stderr)
                raise RuntimeError("bcftools subset failed")
            print(f"Subset created in {time.time()-t0:.1f} s with bcftools")
            VCF_TO_USE = SUBSET_VCF
        else:
            try:
                import pysam
                subset_plain = os.path.join(DATA_DIR, "chr22_5mb.vcf")  # uncompressed; PLINK reads it
                t0 = time.time()
                with pysam.VariantFile(VCF_CHR22) as invcf, \
                     pysam.VariantFile(subset_plain, "w", header=invcf.header) as outvcf:
                    for rec in invcf.fetch("22", 20_000_000, 25_000_000):
                        outvcf.write(rec)
                print(f"Subset created in {time.time()-t0:.1f} s with pysam")
                VCF_TO_USE = subset_plain
            except ModuleNotFoundError:
                raise ModuleNotFoundError(
                    "Need bcftools (brew install bcftools) or pysam (pip install pysam), then re-run."
                ) from None
            except Exception as e:
                print("Subset failed:", e)
                raise
    PLINK_PREFIX = os.path.join(OUT_DIR, "chr22_5mb")
print("VCF_TO_USE:", VCF_TO_USE)
print("PLINK_PREFIX:", PLINK_PREFIX)

Subset already exists: /Users/colecarter/CSE284Project/data/chr22_5mb.vcf.gz
VCF_TO_USE: /Users/colecarter/CSE284Project/data/chr22_5mb.vcf.gz
PLINK_PREFIX: /Users/colecarter/CSE284Project/outputs/chr22_5mb


## 2. Convert VCF to PLINK binary

Uses the VCF you have (data/ or project root). Conversion can take a while on full chr22; quick run is much faster.

In [ ]:
if not os.path.isfile(VCF_TO_USE):
    raise FileNotFoundError(
        "VCF not found. For full chr22 use the full VCF; for quick run run the 'Create 5 Mb subset' cell first."
    )

# Convert VCF to PLINK binary (only if not already done)
if not os.path.isfile(PLINK_PREFIX + ".bed"):
    t0 = time.time()
    cmd = [
        PLINK_CMD, "--vcf", VCF_TO_USE,
        "--chr", "22",
        "--out", PLINK_PREFIX,
        "--make-bed",
        "--allow-extra-chr",
        "--double-id",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
    if result.returncode != 0:
        print("PLINK stderr:", result.stderr)
        print("PLINK stdout:", result.stdout)
        result.check_returncode()
    elif result.stderr:
        print(result.stderr)
    print(f"Conversion time: {time.time() - t0:.1f} s")
else:
    print("PLINK binary already exists, skipping conversion.")

## 3. Simulated phenotype

1000 Genomes has no traits; using a random quantitative phenotype for the comparison.

In [ ]:
fam_path = PLINK_PREFIX + ".fam"
if os.path.isfile(fam_path):
    fam = pd.read_csv(fam_path, sep=r"\s+", header=None,
                      names=["FID", "IID", "father", "mother", "sex", "pheno"])
    n = len(fam)
    np.random.seed(284)
    pheno = np.random.randn(n)  
    pheno_df = fam[["FID", "IID"]].copy()
    pheno_df["pheno"] = pheno
    pheno_file = os.path.join(OUT_DIR, "pheno_quant.txt")
    pheno_df.to_csv(pheno_file, sep="\t", index=False, header=True)
    print(f"Wrote phenotype for {n} samples to {pheno_file}")
else:
    raise FileNotFoundError("Run the VCF→PLINK conversion cell first.")

## 4. Run PLINK linear regression GWAS

In [ ]:
plink_assoc = os.path.join(OUT_DIR, "plink_linear.assoc.linear")
t0 = time.time()
subprocess.run([
    PLINK_CMD, "--bfile", PLINK_PREFIX,
    "--pheno", pheno_file,
    "--linear", "hide-covar",
    "--out", os.path.join(OUT_DIR, "plink_linear"),
    "--allow-extra-chr",
], check=True, cwd=OUT_DIR)
plink_time = time.time() - t0
print(f"PLINK GWAS finished in {plink_time:.1f} s")

## 5. Run GEMMA (LMM) GWAS

GEMMA needs a relatedness matrix. We compute it from the same genotypes, then run the association.

In [ ]:
gemma_cwd = OUT_DIR
bfile_base = os.path.basename(PLINK_PREFIX)
t0 = time.time()
rc = subprocess.run([
    GEMMA_CMD, "-bfile", bfile_base,
    "-gk", "1",  
    "-o", "relatedness",
], cwd=gemma_cwd, capture_output=True, text=True)
if rc.returncode != 0:
    print("GEMMA relatedness stderr:", rc.stderr)
rc.check_returncode()
print(f"Relatedness matrix computed in {time.time() - t0:.1f} s")

In [ ]:
fam = pd.read_csv(PLINK_PREFIX + ".fam", sep=r"\s+", header=None)
pheno_only = os.path.join(OUT_DIR, "pheno_gemma.txt")
fam[5] = pheno_df["pheno"].values  # replace default pheno with our quant trait
fam[[5]].to_csv(pheno_only, sep="\t", index=False, header=False)

t0 = time.time()
subprocess.run([
    GEMMA_CMD, "-bfile", bfile_base,
    "-k", "output/relatedness.cXX.txt",
    "-p", "pheno_gemma.txt",
    "-lmm", "4",  # Wald test
    "-o", "gemma_lmm",
], cwd=gemma_cwd, check=True)
gemma_time = time.time() - t0
print(f"GEMMA LMM GWAS finished in {gemma_time:.1f} s")

## 6. Load association results

In [ ]:
plink_raw = pd.read_csv(plink_assoc, sep=r"\s+")
plink_df = plink_raw[plink_raw["TEST"] == "ADD"][["CHR", "SNP", "BP", "A1", "BETA", "STAT", "P"]].copy()
plink_df = plink_df.rename(columns={"P": "P_plink", "BETA": "BETA_plink"})
print("PLINK: ", plink_df.shape[0], "SNPs")

gemma_assoc = os.path.join(OUT_DIR, "output", "gemma_lmm.assoc.txt")
if os.path.isfile(gemma_assoc):
    gemma_df = pd.read_csv(gemma_assoc, sep="\t")
    # columns: chr, rs, ps, n_miss, allele1, allele2, af, beta, se, l_reml, p_wald (or p_lrt)
    gemma_df = gemma_df.rename(columns={"ps": "BP", "rs": "SNP", "p_wald": "P_gemma", "beta": "BETA_gemma"})
    if "P_gemma" not in gemma_df.columns:
        gemma_df["P_gemma"] = gemma_df.get("p_lrt", gemma_df.get("p_wald", np.nan))
    print("GEMMA: ", gemma_df.shape[0], "SNPs")
else:
    gemma_df = None
    print("GEMMA output not found; run GEMMA cell first.")

## 7. QQ plots

In [ ]:
def qq_plot(p_values, label, ax):
    p = np.array(p_values).flatten()
    p = p[~(p <= 0)]
    p = np.sort(p)
    n = len(p)
    expected = -np.log10(np.linspace(1 / (n + 1), n / (n + 1), n))
    observed = -np.log10(p)
    ax.scatter(expected, observed, s=4, alpha=0.6, label=label)
    lim = max(expected.max(), observed.max()) * 1.05
    ax.plot([0, lim], [0, lim], "k--", lw=1, label="y=x")
    ax.set_xlabel("Expected -log10(p)")
    ax.set_ylabel("Observed -log10(p)")
    ax.legend()
    ax.set_title(f"QQ plot: {label}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
qq_plot(plink_df["P_plink"], "PLINK (linear)", axes[0])
if gemma_df is not None:
    qq_plot(gemma_df["P_gemma"], "GEMMA (LMM)", axes[1])
else:
    axes[1].set_title("GEMMA (LMM) — no results")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "qq_plink_vs_gemma.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Manhattan plots

In [ ]:
def manhattan(p_df, p_col, title, ax):
    p_df = p_df.copy()
    p_df["log10p"] = -np.log10(p_df[p_col].clip(lower=1e-300))
    ax.scatter(p_df["BP"], p_df["log10p"], s=2, alpha=0.6)
    ax.axhline(-np.log10(5e-8), color="red", linestyle="--", linewidth=1, label="5e-8")
    ax.set_xlabel("Position (bp)")
    ax.set_ylabel("-log10(p)")
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
manhattan(plink_df, "P_plink", "PLINK linear regression", axes[0])
if gemma_df is not None:
    manhattan(gemma_df, "P_gemma", "GEMMA LMM", axes[1])
else:
    axes[1].set_title("GEMMA LMM — no results")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "manhattan_plink_vs_gemma.png"), dpi=150, bbox_inches="tight")
plt.show()

## 9. Scatter: PLINK vs GEMMA p-values (and effect sizes)

Merge results by SNP/position to compare concordance.

In [ ]:
if gemma_df is not None:
    merge_col = "SNP" if "SNP" in gemma_df.columns else "BP"
    m = plink_df.merge(gemma_df[["SNP", "BP", "P_gemma", "BETA_gemma"]].drop_duplicates(),
                       on=["SNP", "BP"], how="inner")
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].scatter(-np.log10(m["P_plink"].clip(1e-300)), -np.log10(m["P_gemma"].clip(1e-300)), s=2, alpha=0.5)
    lim = max(axes[0].get_xlim()[1], axes[0].get_ylim()[1])
    axes[0].plot([0, lim], [0, lim], "k--", lw=1)
    axes[0].set_xlabel("-log10(p) PLINK")
    axes[0].set_ylabel("-log10(p) GEMMA")
    axes[0].set_title("P-value comparison")
    axes[1].scatter(m["BETA_plink"], m["BETA_gemma"], s=2, alpha=0.5)
    axes[1].set_xlabel("BETA PLINK")
    axes[1].set_ylabel("BETA GEMMA")
    axes[1].set_title("Effect size comparison")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "scatter_plink_vs_gemma.png"), dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Merge skipped (no GEMMA results).")

## 10. Runtime comparison (optional)

If you ran both tools in this session, we can report runtimes.

In [ ]:
try:
    print(f"PLINK linear GWAS: {plink_time:.1f} s")
    print(f"GEMMA LMM GWAS:   {gemma_time:.1f} s")
except NameError:
    print("Rerun the PLINK and GEMMA cells to see runtimes.")